# 독립 fleet inventory·비용계획·한정 calibration — CPU
RGB269 작업과 다른 새 CPU runtime/Drive 폴더를 사용합니다.90archive/354parts/3.094TB는관측metadata합이며공식조각수대조만완료,내용/runtime version/decoded규모/전체처리량은미검증입니다.
기본ACTION=inventory는원격원본을읽지않습니다. pin은선택archive metadata만64요청/300초까지읽습니다. calibration_plan은4MiB source목표/32MiB decode allowance의명시pilot계획이며전체archive추정이아닙니다. planning priors와측정결과를구분합니다.
고정plan SHA를검토한뒤calibrate를선택하면한segment만수행합니다. Source16MiB/1000HTTP/120초예약중일반softread4MiB,decode32MiB,caspack4MiB이하. 정상actual+명시tail로정산하며hardloss미확인은최대예약을유지합니다. 전체run은unknown추정치를채우고cost×2계획을고정하기전거부합니다.
STORE_DIR은사용자가실행하는Colab에서닫힌불변pack/marker/readback만저장합니다. LiveSQLite는local,분산atomicity/durability보장없음,이전정상자료/실패잔여자동삭제없음. Stale lease/pending예약은이전owner종료를확인한명시token복구만허용합니다.
catalog완료와CRC/fullstream검증을구분합니다. 기본catalog_only는두번째archive검증pass를자동실행하지않습니다. test자료는index만이며학습금지,depth는별도archive로등록합니다. 실제전체index완료/학습을주장하지않습니다.


In [ ]:
# Root가 source/ready revision과 source ZIP SHA를 고정합니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""
NOTEBOOK_REVISION = ""
INPUT_ZIP_SHA256 = "273206d1b13028e8dc66745b30f3e2b10eaf4b07448c58c5fd77bc7ed54b3e99"
CONFIG_SHA256 = "cc933b2897adc813e568b9bb344c50b1e46b9032d8ccaa0366817bb8ce61a562"
SOURCE_ROOT = "/content/fleet_index_v1-source"
INPUT_ROOT = "/content/fleet_index_v1-inputs"
WORKSPACE = "/content/fleet_index_v1-local"
ACTION = "inventory" # inventory / pin / calibration_plan / full_plan / calibrate / index
SELECTED_KEYS = ["UnrealEgoData2_train_val_rgb/ArchVisInterior_ArchVis_RT"]
RUNTIME_INVENTORY_PATH = "" # pin이 출력한 작은 local JSON을 명시합니다.
ESTIMATES_PATH = "" # calibration 근거에서 검토한 전체 추정 JSON. 빈값은 null template.
RECOVERY_DIRECTORY = "/content"
EXPECTED_PLAN_SHA256 = "" # 검토한 고정 plan hash. 빈값이면 원본 run 차단.
STORE_DIR = "" # 별도 /content/drive/MyDrive/.../fleet_index_v1 디렉터리
OWNER = "fleet_calibration_001"
MAX_SEGMENTS = 1 # full은 고정 총예산 내 명시 증가 가능. 자동 budget 갱신 없음.
SESSION_SECONDS = 300


In [ ]:
import hashlib, io, json, re, stat, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SOURCE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", INPUT_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", CONFIG_SHA256)
assert re.fullmatch("[a-f0-9]{40}", SOURCE_REVISION)
assert not Path(WORKSPACE).is_symlink(), "local cache 링크 금지"
import psutil, shutil
assert psutil.virtual_memory().available >= 2*1024**3, "가용 RAM2GiB 필요"
assert shutil.disk_usage(Path(SOURCE_ROOT).parent).free >= 4*1024**3, "가용 disk4GiB 필요"
print("1/2: 고정 source ZIP을 선택하세요.")
source_uploaded = files.upload()
print("2/2: 고정 input ZIP을 선택하세요.")
input_uploaded = files.upload()
def selected(uploaded, pin, label):
    rows = [b for b in uploaded.values() if hashlib.sha256(b).hexdigest() == pin]
    assert len(rows) == 1, label + " ZIP 누락 또는 고정 SHA 불일치"
    return rows[0]
def inspect_zip(raw, maximum):
    assert len(raw) <= maximum, "압축 ZIP 크기 상한"
    payloads, directories, names = {}, set(), set()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        rows = z.infolist()
        assert len(rows) <= 2048 and sum(i.file_size for i in rows) <= maximum, "ZIP 해제 상한"
        for item in rows:
            name = item.filename.rstrip("/")
            path = PurePosixPath(name)
            assert name and name == path.as_posix() and not path.is_absolute()
            assert ".." not in path.parts and "\\" not in name and all(":" not in part for part in path.parts), "ZIP 경로 오류"
            assert name.casefold() not in names, "ZIP 중복 경로"
            names.add(name.casefold())
            kind = stat.S_IFMT(item.external_attr >> 16)
            assert kind in (0, stat.S_IFDIR if item.is_dir() else stat.S_IFREG), "ZIP 링크/특수파일 금지"
            for parent in path.parents:
                if str(parent) != ".": directories.add(parent.as_posix())
            data = z.read(item)  # 모든 member CRC를 추출 전에 검사
            assert len(data) == item.file_size
            if item.is_dir(): directories.add(name)
            else: payloads[name] = data
    assert not set(payloads) & directories, "ZIP 파일/폴더 충돌"
    return payloads, directories
# 두 ZIP 모두 검증하기 전에는 source/input/준비 디렉터리를 만들지 않는다.
source_data = inspect_zip(selected(source_uploaded, SOURCE_ZIP_SHA256, "source"), 64*1024**2)
input_data = inspect_zip(selected(input_uploaded, INPUT_ZIP_SHA256, "input"), 4*1024**2)
assert source_data[0]["SOURCE_REVISION.txt"].decode("utf-8").strip() == SOURCE_REVISION
input_config = json.loads(input_data[0]["config.json"])
config_raw = json.dumps(input_config, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
assert hashlib.sha256(config_raw).hexdigest() == CONFIG_SHA256
def existing_matches(destination, contents):
    target = Path(destination).absolute()
    for part in (target, *target.parents):
        assert not part.is_symlink() and not getattr(part, "is_junction", lambda:False)(), "추출 경로 링크 금지"
    if not target.exists(): return False
    assert target.is_dir(), "추출 대상이 디렉터리가 아님"
    files_seen, dirs_seen = set(), set()
    for count, path in enumerate(target.rglob("*")):
        assert count < 4096 and not path.is_symlink() and not getattr(path, "is_junction", lambda:False)()
        relative = path.relative_to(target).as_posix()
        if path.is_dir(): dirs_seen.add(relative)
        else:
            assert path.is_file() and relative in contents[0], "기존 추출본의 다른 파일"
            expected = contents[0][relative]
            assert path.stat().st_size == len(expected) and path.read_bytes() == expected, "기존 추출본 바이트 불일치"
            files_seen.add(relative)
    assert files_seen == set(contents[0]) and dirs_seen == contents[1], "기존 추출본 파일/폴더 집합 불일치"
    return True
source_path, input_path = Path(SOURCE_ROOT).absolute(), Path(INPUT_ROOT).absolute()
assert not source_path.is_relative_to(input_path) and not input_path.is_relative_to(source_path)
source_exists = existing_matches(SOURCE_ROOT, source_data)
input_exists = existing_matches(INPUT_ROOT, input_data)
def materialize(destination, contents, exists):
    if exists: return  # 정확히 같은 기존 추출본만 읽기 재사용
    target = Path(destination)
    target.mkdir(parents=True, exist_ok=False)
    for name in sorted(contents[1], key=lambda s:len(PurePosixPath(s).parts)):
        (target/name).mkdir(exist_ok=True)
    for name, data in contents[0].items():
        with (target/name).open("xb") as stream: stream.write(data)
materialize(SOURCE_ROOT, source_data, source_exists)
materialize(INPUT_ROOT, input_data, input_exists)
sys.dont_write_bytecode = True
sys.path.insert(0, SOURCE_ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "indexed_gzip==1.10.3", "psutil"])
import indexed_gzip
assert indexed_gzip.__version__ == "1.10.3"
del source_uploaded, input_uploaded, source_data, input_data


In [ ]:
import sys,json
from pathlib import Path
for name,module in tuple(sys.modules.items()):
    if name.startswith(("rewind_fleet", "rewind_archive")):
        loaded=getattr(module,"__file__",None)
        assert loaded and Path(loaded).resolve().is_relative_to(Path(SOURCE_ROOT).resolve()), "이전 코드: 새 runtime 필요"
sys.path.insert(0,SOURCE_ROOT)
from rewind_fleet import planning
from rewind_fleet.__main__ import authorized_factory
from rewind_fleet.storage import Store
INVENTORY=planning.validate(json.loads((Path(INPUT_ROOT)/"inventory.json").read_bytes()),"fleet_inventory_v1")
if RUNTIME_INVENTORY_PATH:
    INVENTORY=planning.validate(json.loads(Path(RUNTIME_INVENTORY_PATH).read_bytes()),"fleet_inventory_v1")
print({"archives":len(INVENTORY["archives"]),"compressed_bytes":INVENTORY["known_compressed_bytes"],
       "official_counts":INVENTORY["official_registry_complete"],"selected":SELECTED_KEYS})
if ACTION == "pin":
    from google.colab import auth
    auth.authenticate_user()
    session=authorized_factory()()
    try:INVENTORY=planning.pin(INVENTORY,session,SELECTED_KEYS,max_requests=64,max_seconds=300)
    finally:session.close()
    Path(WORKSPACE).mkdir(exist_ok=True)
    target=Path(WORKSPACE)/("runtime_inventory_"+INVENTORY["content_sha256"]+".json")
    if not target.exists():target.write_text(json.dumps(INVENTORY,ensure_ascii=False),encoding="utf8")
    print("Pinned inventory",str(target),INVENTORY["metadata_pin_stats"])


In [ ]:
if ACTION in ("calibration_plan","calibrate"):
    assert len(SELECTED_KEYS)==1
    COST_PLAN=planning.calibration_plan(INVENTORY,SELECTED_KEYS[0])
else:
    ESTIMATES=json.loads((Path(ESTIMATES_PATH) if ESTIMATES_PATH else Path(INPUT_ROOT)/"estimates.json").read_bytes())
    COST_PLAN=planning.plan(INVENTORY,ESTIMATES,SELECTED_KEYS,verification="catalog_only")
print({"plan_sha256":COST_PLAN["content_sha256"],"complete_estimate":COST_PLAN["complete_estimate"],
       "base":COST_PLAN["base"],"cap":COST_PLAN["cap"],"missing":COST_PLAN["missing"]})
Path(WORKSPACE).mkdir(exist_ok=True)
PLAN_PATH=Path(WORKSPACE)/("plan_"+COST_PLAN["content_sha256"]+".json")
if not PLAN_PATH.exists():PLAN_PATH.write_text(json.dumps(COST_PLAN,ensure_ascii=False),encoding="utf8")
print("Cost plan",str(PLAN_PATH))


In [ ]:
if ACTION in ("calibrate","index"):
    try:
        assert COST_PLAN["complete_estimate"] and EXPECTED_PLAN_SHA256 == COST_PLAN["content_sha256"], "고정 plan 검토/pin 필요"
        assert (ACTION == "calibrate") == (COST_PLAN["scope"] == "calibration")
        assert STORE_DIR and STORE_DIR.startswith("/content/drive/") and "fleet_index" in Path(STORE_DIR).name, "별도 fleet 저장 디렉터리 필요"
        from google.colab import auth,drive
        auth.authenticate_user()
        drive.mount("/content/drive")
        from rewind_fleet.runner import run
        REPORT=run(INVENTORY,COST_PLAN,STORE_DIR,WORKSPACE,owner=OWNER,single_writer=True,
            max_segments=MAX_SEGMENTS,session_seconds=SESSION_SECONDS,session_factory=authorized_factory())
        print({k:REPORT[k] for k in ("head","stop_reason","reconciled_usage","pending","selected_archives_complete","dataset_complete")})
    except BaseException as exc:
        import uuid
        directory=Path(WORKSPACE);directory.mkdir(exist_ok=True)
        diagnostic={"stage":"run_entry_or_execution","type":type(exc).__name__,"reason":getattr(exc,"reason","unconfirmed"),"original_requested_by_recovery":False}
        target=directory/("fleet_error_"+uuid.uuid4().hex+".json")
        with target.open("x",encoding="utf8") as f:json.dump(diagnostic,f)
        raise


In [ ]:
# 원본 요청/인증/재실행 없이 작은 report·plan만 회수. 전체 index를 ZIP 복제하지 않습니다.
import hashlib,json,zipfile,uuid
from pathlib import Path
from google.colab import files
root=Path(WORKSPACE);chosen=[]
if root.is_dir():
    chosen=sorted([p for p in root.iterdir() if p.is_file() and p.suffix==".json"],key=lambda p:p.stat().st_mtime)[-8:]
assert sum(p.stat().st_size for p in chosen)<=8*1024**2
target=Path(RECOVERY_DIRECTORY)/("fleet_recovery_"+uuid.uuid4().hex+".zip")
recovery_info={"status":"local_reports_and_best_effort_store","raw_archive_requested":False,"live_store":STORE_DIR,
    "lease_auto_released":False,"new_runtime_durability_verified":False}
if STORE_DIR:
    try:
        from rewind_fleet.storage import status
        recovery_info["store_state"]=status(STORE_DIR)
    except Exception as exc:recovery_info["store_status_error"]={"type":type(exc).__name__,"reason":getattr(exc,"reason","unreadable")}
with zipfile.ZipFile(target,"x",zipfile.ZIP_STORED) as z:
    for p in chosen:
        assert not p.is_symlink();z.write(p,p.name)
    z.writestr("recovery.json",json.dumps(recovery_info))
print(str(target),target.stat().st_size,hashlib.sha256(target.read_bytes()).hexdigest())
files.download(str(target))
